# Tiny LightGBM from Scratch

This notebook implements a **small educational version of LightGBM for regression**.

It is deliberately simplified so that the main ideas are visible.

We will focus on two characteristic ideas:

1. **Histogram-based split search**
2. **Leaf-wise tree growth**

We also keep the same boosting logic used by gradient-boosted decision trees:

\[
F_t(x) = F_{t-1}(x) + \eta f_t(x)
\]

For squared-error regression, each new tree tries to correct the current residuals.

> This is **not** a full reimplementation of the LightGBM library.
> It is a minimal model designed to show the algorithmic ideas clearly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# 1. Create a toy regression dataset

We use a one-dimensional dataset so that the behavior of the tree is easy to inspect.

In [ ]:
X = np.linspace(0, 10, 80)

y = (
    np.sin(X)
    + 0.25 * np.cos(2 * X)
    + 0.15 * np.random.randn(len(X))
)

plt.figure(figsize=(8, 4))
plt.scatter(X, y)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Toy regression dataset")
plt.show()

# 2. Histogram binning

A classical decision tree may evaluate many possible split values.

LightGBM speeds this up by converting continuous values into a smaller number of **bins**.

Instead of checking every exact feature value:

```text
0.13, 0.19, 0.24, 0.25, 0.31, ...
```

we group them into bins:

```text
bin 0, bin 1, bin 2, ..., bin K
```

This reduces the number of candidate splits.

In this notebook we use equal-width bins for simplicity.
Real LightGBM uses more sophisticated bin construction.

In [ ]:
def make_bins(X, n_bins=16):
    edges = np.linspace(
        X.min(),
        X.max(),
        n_bins + 1
    )

    # digitize returns bin indices
    bin_ids = np.digitize(
        X,
        edges[1:-1]
    )

    return bin_ids, edges

In [ ]:
bin_ids, edges = make_bins(X, n_bins=12)

print("Bin edges:")
print(edges)

print("\nFirst 20 bin assignments:")
print(bin_ids[:20])

# 3. Split score

For squared error, suppose a candidate split creates:

- left residuals \(r_L\)
- right residuals \(r_R\)

The best constant prediction inside each leaf is simply its mean residual.

So we score a split using the reduction in squared error:

\[
Gain =
SSE_{parent}
-
SSE_{left}
-
SSE_{right}
\]

where:

\[
SSE = \sum_i (r_i - \bar r)^2
\]

The best split is the one with the largest positive gain.

In [ ]:
def squared_error(values):
    if len(values) == 0:
        return 0.0

    mean_value = values.mean()
    return np.sum(
        (values - mean_value) ** 2
    )

# 4. Tree node

In [ ]:
class TreeNode:
    def __init__(
        self,
        indices,
        value,
        depth=0
    ):
        self.indices = indices
        self.value = value
        self.depth = depth

        self.threshold = None
        self.left = None
        self.right = None

        self.gain = 0.0
        self.is_leaf = True

# 5. Histogram-based split search

For one feature, the search works as follows:

1. Bin the feature values.
2. Consider boundaries between bins.
3. Compute the gain for each boundary.
4. Keep the split with maximum gain.

This avoids checking every raw feature value.

In [ ]:
def find_best_histogram_split(
    X,
    residuals,
    indices,
    n_bins=16,
    min_samples_leaf=3
):
    X_node = X[indices]
    r_node = residuals[indices]

    if len(indices) < 2 * min_samples_leaf:
        return None

    bin_ids, edges = make_bins(
        X_node,
        n_bins=n_bins
    )

    parent_error = squared_error(r_node)

    best_gain = 0.0
    best_threshold = None

    unique_bins = np.unique(bin_ids)

    for b in unique_bins[:-1]:

        left_local = bin_ids <= b
        right_local = ~left_local

        if left_local.sum() < min_samples_leaf:
            continue

        if right_local.sum() < min_samples_leaf:
            continue

        left_error = squared_error(
            r_node[left_local]
        )

        right_error = squared_error(
            r_node[right_local]
        )

        gain = (
            parent_error
            - left_error
            - right_error
        )

        if gain > best_gain:
            best_gain = gain

            # Boundary between neighboring bins
            threshold_index = min(
                b + 1,
                len(edges) - 1
            )

            best_threshold = edges[
                threshold_index
            ]

    if best_threshold is None:
        return None

    left_indices = indices[
        X[indices] < best_threshold
    ]

    right_indices = indices[
        X[indices] >= best_threshold
    ]

    return {
        "threshold": best_threshold,
        "gain": best_gain,
        "left_indices": left_indices,
        "right_indices": right_indices
    }

# 6. Leaf-wise growth

Many decision-tree implementations grow trees **level by level**.

LightGBM instead grows the tree **leaf-wise**:

- evaluate all current leaves,
- find the leaf whose best split gives the largest gain,
- split that leaf,
- repeat.

Conceptually:

```text
Start:
        root

After first split:
        root
       /    \
     leaf   leaf

Then LightGBM checks both leaves and splits whichever gives more gain.
```

This can reduce loss quickly, but deep branches may form, so practical LightGBM uses controls such as:

- `num_leaves`
- `max_depth`
- `min_data_in_leaf`

In [ ]:
class HistogramLeafWiseTree:
    def __init__(
        self,
        max_leaves=5,
        n_bins=16,
        min_samples_leaf=3
    ):
        self.max_leaves = max_leaves
        self.n_bins = n_bins
        self.min_samples_leaf = min_samples_leaf

        self.root = None
        self.leaves = []

    def fit(self, X, residuals):

        all_indices = np.arange(len(X))

        root_value = residuals.mean()

        self.root = TreeNode(
            indices=all_indices,
            value=root_value,
            depth=0
        )

        self.leaves = [self.root]

        while len(self.leaves) < self.max_leaves:

            best_leaf = None
            best_split = None
            best_gain = 0.0

            # Evaluate every current leaf
            for leaf in self.leaves:

                split = find_best_histogram_split(
                    X,
                    residuals,
                    leaf.indices,
                    n_bins=self.n_bins,
                    min_samples_leaf=self.min_samples_leaf
                )

                if split is None:
                    continue

                if split["gain"] > best_gain:
                    best_gain = split["gain"]
                    best_leaf = leaf
                    best_split = split

            # No useful split remains
            if best_leaf is None:
                break

            # Create children
            left_indices = best_split[
                "left_indices"
            ]

            right_indices = best_split[
                "right_indices"
            ]

            left_node = TreeNode(
                indices=left_indices,
                value=residuals[
                    left_indices
                ].mean(),
                depth=best_leaf.depth + 1
            )

            right_node = TreeNode(
                indices=right_indices,
                value=residuals[
                    right_indices
                ].mean(),
                depth=best_leaf.depth + 1
            )

            best_leaf.threshold = (
                best_split["threshold"]
            )

            best_leaf.gain = (
                best_split["gain"]
            )

            best_leaf.left = left_node
            best_leaf.right = right_node
            best_leaf.is_leaf = False

            # Replace selected leaf by its two children
            self.leaves.remove(best_leaf)
            self.leaves.extend(
                [left_node, right_node]
            )

        return self

    def _predict_one(self, x):

        node = self.root

        while not node.is_leaf:

            if x < node.threshold:
                node = node.left
            else:
                node = node.right

        return node.value

    def predict(self, X):

        return np.array(
            [self._predict_one(x) for x in X]
        )

# 7. Tiny LightGBM regressor

We now combine the leaf-wise histogram trees in a boosting ensemble.

For squared-error regression:

\[
r_i = y_i - \hat y_i
\]

Each new tree fits the residuals.

The ensemble update is:

\[
F_t(x)
=
F_{t-1}(x)
+
\eta f_t(x)
\]

where \(\eta\) is the learning rate.

In [ ]:
class SimpleLightGBMRegressor:

    def __init__(
        self,
        n_estimators=30,
        learning_rate=0.1,
        max_leaves=5,
        n_bins=16,
        min_samples_leaf=3
    ):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_leaves = max_leaves
        self.n_bins = n_bins
        self.min_samples_leaf = min_samples_leaf

        self.base_score = None
        self.trees = []
        self.training_loss = []

    def fit(self, X, y):

        self.base_score = y.mean()

        predictions = np.full(
            len(y),
            self.base_score,
            dtype=float
        )

        for step in range(
            self.n_estimators
        ):

            residuals = (
                y - predictions
            )

            tree = HistogramLeafWiseTree(
                max_leaves=self.max_leaves,
                n_bins=self.n_bins,
                min_samples_leaf=(
                    self.min_samples_leaf
                )
            )

            tree.fit(
                X,
                residuals
            )

            update = tree.predict(X)

            predictions += (
                self.learning_rate
                * update
            )

            self.trees.append(tree)

            mse = np.mean(
                (y - predictions) ** 2
            )

            self.training_loss.append(
                mse
            )

            print(
                f"Tree {step + 1:2d} | "
                f"MSE={mse:.4f} | "
                f"leaves={len(tree.leaves)}"
            )

        return self

    def predict(self, X):

        predictions = np.full(
            len(X),
            self.base_score,
            dtype=float
        )

        for tree in self.trees:

            predictions += (
                self.learning_rate
                * tree.predict(X)
            )

        return predictions

# 8. Train the model

In [ ]:
model = SimpleLightGBMRegressor(
    n_estimators=35,
    learning_rate=0.12,
    max_leaves=5,
    n_bins=12,
    min_samples_leaf=4
)

model.fit(X, y)

# 9. Final prediction

In [ ]:
X_test = np.linspace(
    0,
    10,
    400
)

y_pred = model.predict(
    X_test
)

plt.figure(figsize=(9, 4))

plt.scatter(
    X,
    y,
    label="training data"
)

plt.plot(
    X_test,
    y_pred,
    linewidth=2,
    label="Tiny LightGBM"
)

plt.xlabel("x")
plt.ylabel("y")
plt.title(
    "Tiny histogram + leaf-wise boosting"
)
plt.legend()
plt.show()

# 10. Training loss

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    range(
        1,
        len(model.training_loss) + 1
    ),
    model.training_loss,
    marker="o"
)

plt.xlabel("Boosting iteration")
plt.ylabel("Mean Squared Error")
plt.title("Training loss")
plt.show()

# 11. Watch the ensemble grow

In [ ]:
def predict_after_n_trees(
    model,
    X,
    n_trees
):
    prediction = np.full(
        len(X),
        model.base_score,
        dtype=float
    )

    for tree in model.trees[
        :n_trees
    ]:
        prediction += (
            model.learning_rate
            * tree.predict(X)
        )

    return prediction

In [ ]:
stages = [0, 1, 3, 10, 20, 35]

for n in stages:

    if n == 0:
        y_stage = np.full(
            len(X_test),
            model.base_score
        )
    else:
        y_stage = (
            predict_after_n_trees(
                model,
                X_test,
                n
            )
        )

    plt.figure(
        figsize=(8, 3.5)
    )

    plt.scatter(
        X,
        y,
        label="data"
    )

    plt.plot(
        X_test,
        y_stage,
        linewidth=2,
        label=f"{n} trees"
    )

    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(
        f"Prediction after {n} trees"
    )
    plt.legend()
    plt.show()

# 12. Inspect the leaves of the first tree

Each leaf stores:

- the training samples that reached it,
- the residual value predicted by the leaf,
- its depth.

Because growth is leaf-wise, leaf depths do not have to be equal.

In [ ]:
first_tree = model.trees[0]

for i, leaf in enumerate(
    first_tree.leaves
):
    print(
        f"Leaf {i}: "
        f"depth={leaf.depth}, "
        f"samples={len(leaf.indices)}, "
        f"value={leaf.value:.4f}"
    )

# 13. Effect of the number of histogram bins

More bins give the tree more candidate split positions.

Fewer bins make split search cheaper but coarser.

In real LightGBM this is one of the reasons histogram-based learning is efficient.

In [ ]:
for n_bins in [4, 8, 16, 32]:

    temp_model = SimpleLightGBMRegressor(
        n_estimators=20,
        learning_rate=0.15,
        max_leaves=5,
        n_bins=n_bins,
        min_samples_leaf=4
    )

    temp_model.fit(
        X,
        y
    )

    prediction = temp_model.predict(
        X_test
    )

    plt.figure(
        figsize=(8, 3.5)
    )

    plt.scatter(
        X,
        y
    )

    plt.plot(
        X_test,
        prediction,
        linewidth=2,
        label=f"{n_bins} bins"
    )

    plt.title(
        f"Histogram resolution: {n_bins} bins"
    )

    plt.legend()
    plt.show()

# 14. Gradient Boosting vs XGBoost vs LightGBM

## Basic Gradient Boosting

Fits each new tree to the current residuals:

\[
r_i = y_i - \hat y_i
\]

Conceptually:

```text
prediction
    |
residuals
    |
fit tree
    |
add tree
```

---

## XGBoost

Uses gradients and Hessians and an explicit regularized split objective:

```text
prediction
    |
gradient + Hessian
    |
split gain
    |
regularized leaf values
    |
add tree
```

---

## LightGBM

Uses gradient boosting too, but focuses heavily on efficient tree construction:

```text
prediction
    |
residual / gradient information
    |
bin continuous values
    |
histogram split search
    |
split best leaf
    |
add leaf-wise tree
```

Two important LightGBM ideas demonstrated in this notebook are:

### Histogram-based splitting

Continuous values are compressed into discrete bins.

This reduces the number of split candidates.

### Leaf-wise growth

Instead of expanding every node at the same depth, LightGBM chooses the current leaf with the largest potential loss reduction.

This often improves the objective faster for a given number of leaves.

# 15. Important LightGBM ideas not implemented here

The real LightGBM system contains many additional techniques.

Examples include:

- multi-feature trees,
- gradient/Hessian histogram accumulation,
- **GOSS**: Gradient-based One-Side Sampling,
- **EFB**: Exclusive Feature Bundling,
- categorical-feature handling,
- missing-value handling,
- L1/L2 regularization,
- row and feature subsampling,
- parallel training,
- distributed training,
- GPU training,
- optimized histogram subtraction.

This notebook intentionally keeps only the core ideas needed to understand why LightGBM differs from a simple Gradient Boosting implementation.

# Main ideas to remember

A LightGBM-style boosted model still has the familiar form:

\[
F_M(x)
=
F_0(x)
+
\eta
\sum_{m=1}^{M}
f_m(x)
\]

The important engineering ideas are how each \(f_m\) is built.

### Histogram search

\[
\text{continuous feature}
\rightarrow
\text{discrete bins}
\]

This reduces candidate split positions.

### Leaf-wise growth

At each step:

\[
\text{split}
=
\arg\max_{\text{current leaves}}
Gain
\]

So the tree grows where the greatest loss reduction is available.

Those two ideas are central to understanding LightGBM.